<a href="https://colab.research.google.com/github/setdaygamer12/Project-Ai-du-doan-chi-tay/blob/APP-d%E1%BB%B1-%C4%91o%C3%A1n-ch%E1%BB%89-tay/Palmistry_App_Ngrok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile app.py
import os
import io
import json
import math
import base64
import zipfile
import urllib.request
from copy import deepcopy

import numpy as np
from PIL import Image, ImageDraw, ImageOps, ImageEnhance, ImageFilter, ImageFont

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import HTMLResponse, JSONResponse

from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


APP_NAME = "PalmVibe"
IMG_SIZE = (224, 224)

ROI_ORDER = ["Sinh_Dao", "Tam_Dao", "Tri_Dao", "Su_Nghiep"]

DISPLAY_NAME = {
    "Sinh_Dao": "Sinh Đạo",
    "Tam_Dao": "Tâm Đạo",
    "Tri_Dao": "Trí Đạo",
    "Su_Nghiep": "Sự Nghiệp",
}

ASCII_NAME = {
    "Sinh_Dao": "Sinh Dao",
    "Tam_Dao": "Tam Dao",
    "Tri_Dao": "Tri Dao",
    "Su_Nghiep": "Su Nghiep",
}

# Giữ biến EN_NAME để tương thích code cũ, nhưng output hiển thị vẫn là tiếng Việt có dấu.
EN_NAME = dict(DISPLAY_NAME)

# File 4 trở xuống dùng tiếng Việt có dấu cho card, matrix, radar và phần phân tích.
UI_NAME = dict(DISPLAY_NAME)

ROI_DESC = {
    "Sinh_Dao": "Năng lượng cá nhân • sức bền • nhịp sống",
    "Tam_Dao": "Cảm xúc • tình cảm • khả năng kết nối",
    "Tri_Dao": "Tư duy • logic • phân tích • sáng tạo",
    "Su_Nghiep": "Định hướng • kỷ luật • mục tiêu dài hạn",
}

ROI_COLOR = {
    "Sinh_Dao": "#7EF0FF",
    "Tam_Dao": "#FF61D8",
    "Tri_Dao": "#8C7BFF",
    "Su_Nghiep": "#69FFB2",
}

CLASS_FALLBACK = {
    "Sinh_Dao": 0,
    "Su_Nghiep": 1,
    "Tam_Dao": 2,
    "Tri_Dao": 3,
}

MODEL_CANDIDATES = [
    "/content/drive/MyDrive/Palmistry AI/Palmistry Ai.h5",
    "/content/drive/MyDrive/Palmistry AI Output/Palmistry Ai.h5",
    "/content/Palmistry Ai.h5",
    "./Palmistry Ai.h5",
    "./model/Palmistry Ai.h5",
]

CLASS_CANDIDATES = [
    "/content/drive/MyDrive/Palmistry AI/class_indices.json",
    "/content/drive/MyDrive/Palmistry AI Output/class_indices.json",
    "/content/class_indices.json",
    "./class_indices.json",
    "./model/class_indices.json",
]

HAND_MODEL_PATH = "/content/hand_landmarker.task"
HAND_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"

DEFAULT_ROI = {
    "Sinh_Dao":  {"u1": 0.00, "u2": 0.54, "v1": 0.18, "v2": 0.96},
    "Tam_Dao":   {"u1": 0.08, "u2": 0.92, "v1": 0.04, "v2": 0.30},
    "Tri_Dao":   {"u1": 0.08, "u2": 0.92, "v1": 0.38, "v2": 0.70},
    "Su_Nghiep": {"u1": 0.38, "u2": 0.66, "v1": 0.10, "v2": 0.96},
}

ROI_SEARCH_PRESETS = {
    "Sinh_Dao": [
        {"u1": 0.00, "u2": 0.58, "v1": 0.15, "v2": 1.00},
        {"u1": 0.00, "u2": 0.54, "v1": 0.18, "v2": 0.96},
        {"u1": 0.00, "u2": 0.52, "v1": 0.20, "v2": 0.98},
        {"u1": 0.02, "u2": 0.55, "v1": 0.24, "v2": 1.00},
        {"u1": 0.00, "u2": 0.48, "v1": 0.18, "v2": 0.96},
        {"u1": 0.00, "u2": 0.50, "v1": 0.22, "v2": 0.92},
    ],
    "Tam_Dao": [
        {"u1": 0.05, "u2": 0.95, "v1": 0.05, "v2": 0.38},
        {"u1": 0.08, "u2": 0.92, "v1": 0.04, "v2": 0.30},
        {"u1": 0.08, "u2": 0.95, "v1": 0.08, "v2": 0.34},
        {"u1": 0.10, "u2": 0.92, "v1": 0.03, "v2": 0.28},
        {"u1": 0.16, "u2": 0.94, "v1": 0.05, "v2": 0.33},
        {"u1": 0.18, "u2": 0.95, "v1": 0.05, "v2": 0.31},
    ],
    "Tri_Dao": [
        {"u1": 0.05, "u2": 0.95, "v1": 0.30, "v2": 0.66},
        {"u1": 0.08, "u2": 0.95, "v1": 0.36, "v2": 0.68},
        {"u1": 0.08, "u2": 0.92, "v1": 0.38, "v2": 0.70},
        {"u1": 0.16, "u2": 0.94, "v1": 0.38, "v2": 0.63},
        {"u1": 0.18, "u2": 0.92, "v1": 0.35, "v2": 0.60},
    ],
    "Su_Nghiep": [
        {"u1": 0.35, "u2": 0.68, "v1": 0.08, "v2": 1.00},
        {"u1": 0.38, "u2": 0.66, "v1": 0.10, "v2": 0.96},
        {"u1": 0.40, "u2": 0.65, "v1": 0.14, "v2": 0.98},
        {"u1": 0.36, "u2": 0.67, "v1": 0.08, "v2": 0.95},
    ],
}

RELATED_POSTS = [
    {
        "title": "Hiểu bản thân qua Big Five",
        "tag": "Tính cách",
        "desc": "Một góc nhìn khoa học hơn để người dùng tự soi lại tính cách, điểm mạnh và cách phản ứng của mình.",
        "url": "https://www.verywellmind.com/the-big-five-personality-dimensions-2795422",
        "img": "https://images.unsplash.com/photo-1499750310107-5fef28a66643?auto=format&fit=crop&w=900&q=80",
    },
    {
        "title": "Khám phá hướng nghề nghiệp",
        "tag": "Sự nghiệp",
        "desc": "Công cụ tham khảo để người dùng tìm kiểu công việc hợp sở thích và năng lượng cá nhân.",
        "url": "https://www.onetinterestprofiler.org/",
        "img": "https://images.unsplash.com/photo-1521737604893-d14cc237f11d?auto=format&fit=crop&w=900&q=80",
    },
    {
        "title": "Đặt mục tiêu rõ hơn",
        "tag": "Phát triển",
        "desc": "Gợi ý cách biến kết quả phân tích thành mục tiêu nhỏ, dễ theo dõi và thực tế hơn.",
        "url": "https://www.mindtools.com/a4wo118/smart-goals/",
        "img": "https://images.unsplash.com/photo-1516321497487-e288fb19713f?auto=format&fit=crop&w=900&q=80",
    },
    {
        "title": "Cảm xúc và kết nối",
        "tag": "Tình duyên",
        "desc": "Một danh mục đọc thêm về cảm xúc, điều tiết cảm xúc và cách xây dựng kết nối lành mạnh.",
        "url": "https://greatergood.berkeley.edu/tag/emotional%2Bintelligence",
        "img": "https://images.unsplash.com/photo-1506126613408-eca07ce68773?auto=format&fit=crop&w=900&q=80",
    },
]


def first_existing(paths):
    for path in paths:
        if os.path.exists(path):
            return path
    return None


def clamp_box(x1, y1, x2, y2, w, h):
    x1 = max(0, min(int(round(x1)), w - 1))
    y1 = max(0, min(int(round(y1)), h - 1))
    x2 = max(1, min(int(round(x2)), w))
    y2 = max(1, min(int(round(y2)), h))

    if x2 <= x1:
        x2 = min(w, x1 + 2)
    if y2 <= y1:
        y2 = min(h, y1 + 2)

    return x1, y1, x2, y2


def pil_to_data_url(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("utf-8")


def parse_settings(settings_raw):
    return {
        "roi": deepcopy(DEFAULT_ROI),
        "preset": "Auto",
    }


def load_ui_font(size=24, bold=False):
    """Load a Unicode font so Vietnamese labels render correctly on ROI overlays."""
    candidates = []
    if bold:
        candidates.extend([
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
            "/content/DejaVuSans-Bold.ttf",
        ])
    candidates.extend([
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/content/DejaVuSans.ttf",
    ])

    for path in candidates:
        try:
            if os.path.exists(path):
                return ImageFont.truetype(path, size=size)
        except Exception:
            pass

    return ImageFont.load_default()


class PalmVibeEngine:
    def __init__(self):
        self.model = None
        self.model_ok = False
        self.model_path = first_existing(MODEL_CANDIDATES)

        self.class_ok = False
        self.class_path = first_existing(CLASS_CANDIDATES)
        self.class_to_index = dict(CLASS_FALLBACK)
        self.index_to_class = {v: k for k, v in self.class_to_index.items()}

        self.mp_ok = False
        self.mp = None
        self.detector = None

        self.setup_classes()
        self.setup_model()
        self.setup_mediapipe()

    def setup_classes(self):
        try:
            if self.class_path:
                with open(self.class_path, "r", encoding="utf-8") as f:
                    class_indices = json.load(f)
                self.class_to_index = {k: int(v) for k, v in class_indices.items()}
                self.index_to_class = {int(v): k for k, v in class_indices.items()}
                self.class_ok = True
            else:
                self.class_ok = False
        except Exception:
            self.class_to_index = dict(CLASS_FALLBACK)
            self.index_to_class = {v: k for k, v in self.class_to_index.items()}
            self.class_ok = False

    def setup_model(self):
        try:
            if self.model_path:
                self.model = load_model(self.model_path)
                self.model_ok = True
            else:
                self.model_ok = False
        except Exception:
            self.model = None
            self.model_ok = False

    def setup_mediapipe(self):
        try:
            import mediapipe as mp
            from mediapipe.tasks import python
            from mediapipe.tasks.python import vision

            if not os.path.exists(HAND_MODEL_PATH):
                urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL_PATH)

            base_options = python.BaseOptions(model_asset_path=HAND_MODEL_PATH)
            options = vision.HandLandmarkerOptions(
                base_options=base_options,
                num_hands=1,
                running_mode=vision.RunningMode.IMAGE,
                min_hand_detection_confidence=0.35,
                min_hand_presence_confidence=0.35,
                min_tracking_confidence=0.35,
            )

            self.detector = vision.HandLandmarker.create_from_options(options)
            self.mp = mp
            self.mp_ok = True
        except Exception:
            self.detector = None
            self.mp = None
            self.mp_ok = False

    def status(self):
        return {
            "app": APP_NAME,
            "model_ok": self.model_ok,
            "mediapipe_ok": self.mp_ok,
            "class_ok": self.class_ok,
            "model_path": self.model_path or "missing",
            "class_path": self.class_path or "fallback",
        }

    def detect_landmarks(self, img_pil):
        if not self.mp_ok or self.detector is None:
            return None

        arr = np.array(img_pil.convert("RGB"))
        mp_image = self.mp.Image(image_format=self.mp.ImageFormat.SRGB, data=arr)
        result = self.detector.detect(mp_image)

        if not result.hand_landmarks:
            return None

        h, w, _ = arr.shape
        points = []

        for lm in result.hand_landmarks[0]:
            points.append([int(lm.x * w), int(lm.y * h)])

        if len(points) != 21:
            return None

        return np.array(points, dtype=np.int32)

    def rotate_hand_upright(self, img_pil):
        points = self.detect_landmarks(img_pil)
        if points is None:
            return None, None, 0.0

        wrist = points[0]
        middle_mcp = points[9]

        dx = middle_mcp[0] - wrist[0]
        dy = middle_mcp[1] - wrist[1]

        current_angle = math.degrees(math.atan2(dy, dx))
        rotate_angle = -90 - current_angle

        rotated = img_pil.rotate(rotate_angle, expand=True, fillcolor=(255, 255, 255))
        rotated_points = self.detect_landmarks(rotated)

        if rotated_points is None:
            return img_pil, points, 0.0

        return rotated, rotated_points, rotate_angle

    def hand_orientation(self, points):
        thumb_x = points[2][0]
        pinky_x = points[17][0]

        if thumb_x < pinky_x:
            return {
                "mode": "thumb_left",
                "mode_vi": "Ngón cái ở bên trái",
                "label": "Thumb is left of pinky. Normal u-axis.",
                "label_vi": "Hệ tọa độ bàn tay đang ở hướng thuận.",
                "flip_u": False,
            }

        return {
            "mode": "thumb_right",
            "mode_vi": "Ngón cái ở bên phải",
            "label": "Thumb is right of pinky. Mirrored u-axis.",
            "label_vi": "Ảnh bàn tay bị đảo hướng nên hệ tọa độ đã được hiệu chỉnh.",
            "flip_u": True,
        }

    def create_palm_box(self, img_pil, points):
        w, h = img_pil.size

        wrist = points[0]
        thumb_mcp = points[2]
        index_mcp = points[5]
        middle_mcp = points[9]
        ring_mcp = points[13]
        pinky_mcp = points[17]

        palm_points = np.array([
            wrist,
            thumb_mcp,
            index_mcp,
            middle_mcp,
            ring_mcp,
            pinky_mcp,
        ])

        x1 = np.min(palm_points[:, 0])
        x2 = np.max(palm_points[:, 0])

        y1 = np.min([
            index_mcp[1],
            middle_mcp[1],
            ring_mcp[1],
            pinky_mcp[1],
        ])
        y2 = wrist[1]

        if y2 < y1:
            y1, y2 = y2, y1

        bw = x2 - x1
        bh = y2 - y1

        x1 = x1 - 0.18 * bw
        x2 = x2 + 0.18 * bw
        y1 = y1 - 0.12 * bh
        y2 = y2 + 0.12 * bh

        return clamp_box(x1, y1, x2, y2, w, h)

    def crop_roi_from_box(self, img_pil, points, palm_box, u1, u2, v1, v2):
        left, top, right, bottom = palm_box
        palm_w = right - left
        palm_h = bottom - top

        orientation = self.hand_orientation(points)

        if not orientation["flip_u"]:
            x1 = left + u1 * palm_w
            x2 = left + u2 * palm_w
        else:
            x1 = right - u2 * palm_w
            x2 = right - u1 * palm_w

        y1 = top + v1 * palm_h
        y2 = top + v2 * palm_h

        w, h = img_pil.size
        box = clamp_box(x1, y1, x2, y2, w, h)

        crop = img_pil.crop(box).resize(IMG_SIZE)
        return crop, box

    def predict_crop(self, crop_img):
        img = crop_img.convert("RGB").resize(IMG_SIZE)
        arr = np.array(img)
        batch = np.expand_dims(arr, axis=0)
        batch = preprocess_input(batch)
        pred = self.model.predict(batch, verbose=0)[0]
        return pred

    def enhance_roi(self, crop_img):
        # Giữ xử lý ảnh vừa phải: không đẩy sáng/contrast quá lố để tránh mất vân tay khi camera hơi chói.
        gray = ImageOps.grayscale(crop_img)
        gray = ImageOps.autocontrast(gray, cutoff=1)
        gray = ImageEnhance.Contrast(gray).enhance(1.32)
        gray = ImageEnhance.Sharpness(gray).enhance(1.18)
        gray = gray.filter(ImageFilter.UnsharpMask(radius=1.0, percent=105, threshold=3))
        return gray

    def edge_map(self, crop_img):
        gray = self.enhance_roi(crop_img)
        arr = np.array(gray).astype("float32") / 255.0

        gx = np.zeros_like(arr)
        gy = np.zeros_like(arr)

        gx[:, 1:-1] = np.abs(arr[:, 2:] - arr[:, :-2])
        gy[1:-1, :] = np.abs(arr[2:, :] - arr[:-2, :])

        grad = np.sqrt(gx ** 2 + gy ** 2)
        if grad.max() > 0:
            grad = grad / grad.max()

        return (grad * 255).astype("uint8")

    def roi_features(self, crop_img):
        gray = self.enhance_roi(crop_img)
        arr = np.array(gray).astype("float32") / 255.0
        edge = self.edge_map(crop_img).astype("float32") / 255.0

        contrast = float(np.std(arr))
        threshold = np.percentile(arr, 35)
        dark_density = float(np.mean(arr < threshold))
        edge_strength = float(np.mean(edge))

        diff_x = np.mean(np.abs(np.diff(arr, axis=1)))
        diff_y = np.mean(np.abs(np.diff(arr, axis=0)))
        sharpness = float(diff_x + diff_y)

        brightness = float(np.mean(arr))

        return {
            "brightness": brightness,
            "contrast": contrast,
            "dark_density": dark_density,
            "edge_strength": edge_strength,
            "sharpness": sharpness,
        }

    def norm_0_10(self, value, low, high):
        score = (value - low) / (high - low + 1e-8) * 10
        score = max(0, min(10, score))
        return float(score)

    def image_score(self, crop_img):
        f = self.roi_features(crop_img)

        contrast_score = self.norm_0_10(f["contrast"], 0.05, 0.20)
        density_score = self.norm_0_10(f["dark_density"], 0.15, 0.45)
        edge_score = self.norm_0_10(f["edge_strength"], 0.02, 0.12)
        sharp_score = self.norm_0_10(f["sharpness"], 0.015, 0.09)

        brightness_score = 10 - abs(f["brightness"] - 0.58) / 0.58 * 10
        brightness_score = max(0, min(10, brightness_score))

        image_score = (
            0.25 * contrast_score +
            0.20 * density_score +
            0.25 * edge_score +
            0.20 * sharp_score +
            0.10 * brightness_score
        )

        return {
            "image_score": float(image_score),
            "features": f,
            "components": {
                "contrast": float(contrast_score),
                "density": float(density_score),
                "edge": float(edge_score),
                "sharpness": float(sharp_score),
                "brightness": float(brightness_score),
            }
        }

    def raw_ai_detail(self, pred, expected_roi):
        expected_idx = self.class_to_index.get(expected_roi, 0)
        expected_conf = float(pred[expected_idx] * 100)

        top_idx = int(np.argmax(pred))
        top_key = self.index_to_class.get(top_idx, expected_roi)
        top_conf = float(np.max(pred) * 100)

        p = pred + 1e-8
        entropy = -np.sum(p * np.log(p))
        max_entropy = np.log(len(pred))
        certainty = 1 - entropy / max_entropy
        certainty = float(max(0, min(1, certainty)) * 100)

        return {
            "expected_confidence": expected_conf,
            "top_key": top_key,
            "top_label": DISPLAY_NAME.get(top_key, top_key),
            "top_label_ascii": UI_NAME.get(top_key, top_key),
            "top_confidence": top_conf,
            "certainty": certainty,
            "matched": top_key == expected_roi,
        }

    def calibrate_prediction(self, pred, expected_roi, image_score):
        pred = np.array(pred).astype("float64")
        pred = pred / (pred.sum() + 1e-8)

        expected_idx = self.class_to_index.get(expected_roi, 0)
        one_hot = np.zeros_like(pred)
        one_hot[expected_idx] = 1.0

        quality = max(0.0, min(1.0, image_score / 10.0))

        if expected_roi in ["Sinh_Dao", "Tam_Dao"]:
            base_prior = 0.78
        else:
            base_prior = 0.66

        prior = base_prior + 0.10 * quality
        prior = max(0.62, min(0.88, prior))

        calibrated = (1.0 - prior) * pred + prior * one_hot
        calibrated = calibrated / (calibrated.sum() + 1e-8)
        return calibrated

    def calibrated_ai_detail(self, calibrated_pred, raw_pred, expected_roi):
        expected_idx = self.class_to_index.get(expected_roi, 0)
        expected_conf = float(calibrated_pred[expected_idx] * 100)

        top_idx = int(np.argmax(calibrated_pred))
        top_key = self.index_to_class.get(top_idx, expected_roi)
        top_conf = float(np.max(calibrated_pred) * 100)

        raw = self.raw_ai_detail(raw_pred, expected_roi)

        p = calibrated_pred + 1e-8
        entropy = -np.sum(p * np.log(p))
        max_entropy = np.log(len(p))
        certainty = 1 - entropy / max_entropy
        certainty = float(max(0, min(1, certainty)) * 100)

        ai_score = 0.78 * (expected_conf / 10.0) + 0.22 * (certainty / 10.0)
        ai_score = float(max(0, min(10, ai_score)))

        return {
            "roi_confidence": expected_conf,
            "top_key": top_key,
            "top_label": DISPLAY_NAME.get(top_key, top_key),
            "top_label_ascii": UI_NAME.get(top_key, top_key),
            "top_confidence": top_conf,
            "certainty": certainty,
            "ai_score": ai_score,
            "raw_expected_confidence": raw["expected_confidence"],
            "raw_top_key": raw["top_key"],
            "raw_top_label": raw["top_label"],
            "raw_top_label_ascii": raw["top_label_ascii"],
            "raw_top_confidence": raw["top_confidence"],
            "raw_matched": raw["matched"],
        }

    def final_roi_score(self, crop_img, raw_pred, calibrated_pred, expected_roi):
        image_result = self.image_score(crop_img)
        image_score = float(image_result["image_score"])

        ai = self.calibrated_ai_detail(calibrated_pred, raw_pred, expected_roi)
        ai_score = float(ai["ai_score"])

        if expected_roi in ["Sinh_Dao", "Tam_Dao"] and ai["raw_expected_confidence"] < 20:
            final_score = 0.58 * image_score + 0.42 * ai_score
        else:
            final_score = 0.48 * image_score + 0.52 * ai_score

        final_score = max(0, min(10, final_score))

        return {
            "score": float(final_score),
            "image_score": image_score,
            "image_result": image_result,
            "ai_result": ai,
        }

    def level(self, score):
        if score >= 8.0:
            return "Rõ"
        if score >= 6.3:
            return "Khá rõ"
        if score >= 4.5:
            return "Trung bình"
        return "Mờ"

    def level_ascii(self, score):
        if score >= 8.0:
            return "Clear"
        if score >= 6.3:
            return "Good"
        if score >= 4.5:
            return "Medium"
        return "Faint"

    def rescue_best_roi_crop(self, roi_name, img_pil, points, palm_box, user_layout):
        candidates = []
        user_r = user_layout.get(roi_name)

        if user_r:
            candidates.append(user_r)

        for cand in ROI_SEARCH_PRESETS[roi_name]:
            candidates.append(cand)

        expected_idx = self.class_to_index.get(roi_name, 0)

        best = None
        seen = set()

        for cand in candidates:
            key = tuple(round(cand[k], 3) for k in ["u1", "u2", "v1", "v2"])
            if key in seen:
                continue
            seen.add(key)

            crop, box = self.crop_roi_from_box(
                img_pil,
                points,
                palm_box,
                cand["u1"],
                cand["u2"],
                cand["v1"],
                cand["v2"],
            )

            pred = self.predict_crop(crop)
            image = self.image_score(crop)

            expected_conf = float(pred[expected_idx] * 100)
            raw_top = float(np.max(pred) * 100)

            rank = (
                0.35 * expected_conf +
                7.2 * image["image_score"] +
                0.08 * raw_top
            )

            item = {
                "crop": crop,
                "box": box,
                "pred": pred,
                "image": image,
                "expected_conf": expected_conf,
                "rank": float(rank),
                "params": cand,
            }

            if best is None or item["rank"] > best["rank"]:
                best = item

        return best

    def auto_crop_4_roi(self, img_pil, points, palm_box, user_layout):
        crops = {}
        boxes = {}
        raw_preds = {}
        chosen_params = {}

        for roi_name in ROI_ORDER:
            best = self.rescue_best_roi_crop(
                roi_name=roi_name,
                img_pil=img_pil,
                points=points,
                palm_box=palm_box,
                user_layout=user_layout,
            )

            crops[roi_name] = best["crop"]
            boxes[roi_name] = best["box"]
            raw_preds[roi_name] = best["pred"]
            chosen_params[roi_name] = best["params"]

        return crops, boxes, raw_preds, chosen_params

    def draw_overlay(self, rotated, points, palm_box, roi_boxes):
        out = rotated.copy()
        draw = ImageDraw.Draw(out)

        title_font = load_ui_font(24, bold=True)
        label_font = load_ui_font(22, bold=True)

        draw.rectangle(palm_box, outline="#FFC75A", width=5)
        # File 3 / ảnh overlay: dùng tiếng Việt không dấu để tránh lỗi font khi render canvas/PIL.
        draw.text(
            (palm_box[0], max(0, palm_box[1] - 32)),
            "Vung long ban tay",
            fill="#FFC75A",
            font=title_font,
            stroke_width=2,
            stroke_fill=(5, 8, 18),
        )

        for point in points:
            x, y = int(point[0]), int(point[1])
            draw.ellipse((x - 4, y - 4, x + 4, y + 4), fill="#69FFB2")

        draw.line([tuple(points[0]), tuple(points[9])], fill="#FFFFFF", width=4)

        # File 3 / ảnh overlay: các nhãn vùng crop dùng không dấu: Sinh Dao, Tam Dao, Tri Dao, Su Nghiep.
        for roi_name in ROI_ORDER:
            color = ROI_COLOR[roi_name]
            box = roi_boxes[roi_name]
            label = ASCII_NAME[roi_name]
            draw.rectangle(box, outline=color, width=5)
            draw.text(
                (box[0], max(0, box[1] - 30)),
                label,
                fill=color,
                font=label_font,
                stroke_width=2,
                stroke_fill=(5, 8, 18),
            )

        return out


    def domain_scores(self, scores):
        sinh = scores["Sinh_Dao"]
        tam = scores["Tam_Dao"]
        tri = scores["Tri_Dao"]
        nghe = scores["Su_Nghiep"]
        avg = (sinh + tam + tri + nghe) / 4.0

        return {
            "Sức khỏe": float(0.70 * sinh + 0.20 * tri + 0.10 * avg),
            "Tình cảm": float(0.75 * tam + 0.15 * tri + 0.10 * avg),
            "Tư duy": float(tri),
            "Sự nghiệp": float(0.70 * nghe + 0.20 * tri + 0.10 * avg),
        }

    def build_interpretation(self, summary):
        scores = summary["scores"]
        domains = summary["domains"]

        def quality_line(name, score):
            if score >= 8:
                return f"{name} đang là vùng nổi bật: đường nét rõ, tín hiệu ảnh tốt và đủ cơ sở để tạo phân tích tham khảo."
            if score >= 6.3:
                return f"{name} ở mức khá: có tín hiệu rõ, nhưng vẫn nên xem cùng các vùng khác để tránh kết luận một chiều."
            if score >= 4.5:
                return f"{name} ở mức trung bình: có tín hiệu nhưng chưa thật sự mạnh, kết quả nên dùng như gợi ý nhẹ."
            return f"{name} còn mờ: ảnh hoặc đường chỉ tay vùng này chưa đủ rõ, nên tránh suy diễn quá sâu."

        dominant = summary["dominant_label"]
        weakest_key = min(scores, key=scores.get)
        weakest = DISPLAY_NAME[weakest_key]

        return {
            "Tổng quan": [
                f"PalmVibe ghi nhận vùng nổi bật nhất là {dominant}, cho thấy đây là phần có tín hiệu rõ nhất trong lần quét này.",
                f"Tổng điểm hiện tại đạt {summary['overall']}/10. Mức này đủ tốt để tạo một bản phân tích tham khảo nếu ảnh lòng bàn tay rõ và ổn định.",
                f"Vùng cần đọc cẩn thận hơn là {weakest}. Điểm thấp không có nghĩa là xấu, mà thường cho thấy đường nét vùng đó chưa nổi bật/mờ trong ảnh.",
                "Kết quả nên được xem như một trải nghiệm AI học tập và giải trí, không phải kết luận chắc chắn về tương lai.",
            ],
            "Tính cách": [
                quality_line("Trí Đạo", scores["Tri_Dao"]),
                "Nếu Trí Đạo cao, người dùng thường có xu hướng suy nghĩ kỹ, thích hiểu bản chất vấn đề và quan sát trước khi quyết định.",
                "Kiểu người này thường hợp với cách học theo dự án, tự thử nghiệm, sửa lỗi và cải tiến từng bước.",
                "Nếu Trí Đạo thấp hơn kỳ vọng, nên hiểu là vùng ảnh chưa đủ rõ, không phải kết luận rằng khả năng tư duy yếu.",
            ],
            "Tình duyên": [
                quality_line("Tâm Đạo", scores["Tam_Dao"]),
                "Tâm Đạo được dùng để tham khảo cách người dùng xử lý cảm xúc, mức độ mở lòng và khả năng giữ sự ổn định trong quan hệ.",
                "Điểm Tâm Đạo khá hoặc cao gợi ý người dùng có xu hướng quan tâm cảm xúc, nhưng vẫn cần thời gian để tin tưởng và bộc lộ sâu hơn.",
                "Nếu Tâm Đạo thấp, nên hiểu là tín hiệu ảnh vùng này chưa rõ; app không kết luận chắc chắn về tình yêu hay tương lai quan hệ.",
            ],
            "Tư duy": [
                f"Điểm tư duy hiện tại là {domains['Tư duy']}/10, lấy chủ yếu từ vùng Trí Đạo.",
                "Điểm cao cho thấy ảnh đang thể hiện tốt các đường liên quan đến phân tích, tập trung, học hỏi và ra quyết định.",
                "Phần này nên được diễn giải như một gợi ý về phong cách học và cách xử lý vấn đề, không phải chỉ số IQ.",
                "Nếu người dùng đang phân vân hướng học tập hoặc nghề nghiệp, Trí Đạo cao thường hợp với các lĩnh vực cần quan sát, logic và kiên nhẫn sửa lỗi.",
            ],
            "Sự nghiệp": [
                quality_line("Sự Nghiệp", scores["Su_Nghiep"]),
                f"Điểm sự nghiệp tổng hợp hiện tại là {domains['Sự nghiệp']}/10, có kết hợp thêm Trí Đạo để phản ánh khả năng định hướng và theo đuổi mục tiêu.",
                "Nếu vùng này cao, người dùng hợp với lộ trình cần kỷ luật, tích lũy kỹ năng và xây dựng năng lực từng bước.",
                "Nếu vùng này thấp, nên xem như tín hiệu định hướng chưa rõ trong ảnh hiện tại, không phải dự đoán rằng sự nghiệp sẽ kém.",
            ],
            "Gợi ý phát triển": [
                "Hãy xem kết quả như một tấm gương phản chiếu nhẹ: vùng nào cao thì xem như điểm mạnh hiện tại, vùng nào thấp thì xem như phần cần quan sát thêm.",
                "Nếu Trí Đạo và Sự Nghiệp cùng cao, người dùng nên thử các hoạt động cần phân tích, kỹ thuật, lập kế hoạch hoặc tự học dài hạn.",
                "Nếu Tâm Đạo nổi bật, người dùng nên chú ý phát triển kỹ năng giao tiếp, lắng nghe và cân bằng cảm xúc khi làm việc nhóm.",
                "Ứng dụng dùng cho mục đích học tập và giải trí, không thay thế tư vấn y khoa, tâm lý hoặc nghề nghiệp.",
            ],
        }

    def analyze_palm_image(self, img_pil, source_name="upload", settings=None):
        if settings is None:
            settings = {
                "roi": deepcopy(DEFAULT_ROI),
                "preset": "Auto",
            }

        if not self.mp_ok:
            return {
                "ok": False,
                "error": "MediaPipe is not available. Install mediapipe and restart the server.",
            }

        if not self.model_ok or self.model is None:
            return {
                "ok": False,
                "error": "Palmistry Ai.h5 was not found or could not be loaded.",
            }

        img_pil = img_pil.convert("RGB")

        rotated, points, rotate_angle = self.rotate_hand_upright(img_pil)
        if points is None:
            return {
                "ok": False,
                "error": "Không phát hiện đủ landmark bàn tay. Ảnh cần thấy rõ lòng bàn tay và cổ tay.",
            }

        palm_box = self.create_palm_box(rotated, points)
        orientation = self.hand_orientation(points)

        crops, roi_boxes, raw_preds, chosen_params = self.auto_crop_4_roi(
            img_pil=rotated,
            points=points,
            palm_box=palm_box,
            user_layout=settings["roi"],
        )

        overlay = self.draw_overlay(rotated, points, palm_box, roi_boxes)

        class_order = ["Sinh_Dao", "Su_Nghiep", "Tam_Dao", "Tri_Dao"]

        roi_cards = []
        raw_heatmap = []
        calibrated_heatmap = []
        scores = {}
        quality_scores = []

        for roi_name in ROI_ORDER:
            raw_pred = raw_preds[roi_name]
            image_result = self.image_score(crops[roi_name])

            calibrated_pred = self.calibrate_prediction(
                pred=raw_pred,
                expected_roi=roi_name,
                image_score=image_result["image_score"],
            )

            score_result = self.final_roi_score(
                crop_img=crops[roi_name],
                raw_pred=raw_pred,
                calibrated_pred=calibrated_pred,
                expected_roi=roi_name,
            )

            scores[roi_name] = score_result["score"]
            quality_scores.append(score_result["image_score"])

            ai = score_result["ai_result"]

            raw_row = []
            calibrated_row = []

            for cls in class_order:
                idx = self.class_to_index.get(cls, 0)
                raw_row.append(round(float(raw_pred[idx] * 100), 1))
                calibrated_row.append(round(float(calibrated_pred[idx] * 100), 1))

            raw_heatmap.append(raw_row)
            calibrated_heatmap.append(calibrated_row)

            roi_cards.append({
                "key": roi_name,
                "label": DISPLAY_NAME[roi_name],
                "label_vi": DISPLAY_NAME[roi_name],
                "description": ROI_DESC[roi_name],
                "crop": pil_to_data_url(crops[roi_name]),
                "score": round(score_result["score"], 1),
                "level": self.level(score_result["score"]),
                "level_vi": self.level(score_result["score"]),
                "image_score": round(score_result["image_score"], 1),
                "ai_score": round(ai["ai_score"], 1),
                "roi_confidence": round(ai["roi_confidence"], 1),
                "certainty": round(ai["certainty"], 1),
                "raw_model_signal": ai["raw_top_label"],
                "raw_top_confidence": round(ai["raw_top_confidence"], 1),
                "raw_expected_confidence": round(ai["raw_expected_confidence"], 1),
            })

        overall = round(float(np.mean(list(scores.values()))), 1)
        dominant_key = max(scores, key=scores.get)
        domains = self.domain_scores(scores)

        summary = {
            "overall": overall,
            "dominant_key": dominant_key,
            "dominant_label": DISPLAY_NAME[dominant_key],
            "dominant_label_ascii": UI_NAME[dominant_key],
            "scores": {k: round(v, 1) for k, v in scores.items()},
            "domains": {k: round(v, 1) for k, v in domains.items()},
        }

        interpretation = self.build_interpretation(summary)

        return {
            "ok": True,
            "source_name": source_name,
            "preset": settings.get("preset", "Auto"),
            "orientation": orientation,
            "rotation_angle": round(float(rotate_angle), 2),
            "palm_box": list(map(int, palm_box)),
            "summary": summary,
            "images": {
                "original": pil_to_data_url(img_pil),
                "rotated_overlay": pil_to_data_url(overlay),
            },
            "rois": roi_cards,
            "heatmap": {
                "rows": [DISPLAY_NAME[k] for k in ROI_ORDER],
                "cols": [DISPLAY_NAME[k] for k in class_order],
                "data": calibrated_heatmap,
                "raw_data": raw_heatmap,
            },
            "radar": {
                "labels": ["Sức khỏe", "Tình cảm", "Tư duy", "Sự nghiệp", "Độ rõ ảnh", "Độ chắc AI"],
                "values": [
                    round(domains["Sức khỏe"], 1),
                    round(domains["Tình cảm"], 1),
                    round(domains["Tư duy"], 1),
                    round(domains["Sự nghiệp"], 1),
                    round(float(np.mean(quality_scores)), 1),
                    round(float(np.mean([r["certainty"] for r in roi_cards]) / 10.0), 1),
                ],
            },
            "interpretation": interpretation,
            "related_posts": RELATED_POSTS,
        }


engine = PalmVibeEngine()
app = FastAPI(title="PalmVibe")


@app.get("/api/status")
def api_status():
    return engine.status()


@app.post("/api/analyze")
async def api_analyze(
    file: UploadFile = File(...),
    settings: str = Form(None),
):
    try:
        raw = await file.read()
        img = Image.open(io.BytesIO(raw)).convert("RGB")
        parsed_settings = parse_settings(settings)

        result = engine.analyze_palm_image(
            img_pil=img,
            source_name=file.filename or "upload",
            settings=parsed_settings,
        )

        return JSONResponse(result)

    except Exception as e:
        return JSONResponse({
            "ok": False,
            "error": f"Image processing error: {str(e)}",
        })


@app.post("/api/analyze_batch")
async def api_analyze_batch(
    files: list[UploadFile] = File(...),
    settings: str = Form(None),
):
    parsed_settings = parse_settings(settings)
    rows = []

    for f in files:
        try:
            raw = await f.read()

            if f.filename.lower().endswith(".zip"):
                with zipfile.ZipFile(io.BytesIO(raw), "r") as z:
                    for name in z.namelist():
                        if not name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                            continue

                        try:
                            img = Image.open(io.BytesIO(z.read(name))).convert("RGB")
                            result = engine.analyze_palm_image(img, name, parsed_settings)

                            if result["ok"]:
                                rows.append({
                                    "file": name,
                                    "status": "ok",
                                    "overall": result["summary"]["overall"],
                                    "dominant": result["summary"]["dominant_label"],
                                    "sinh": result["summary"]["scores"]["Sinh_Dao"],
                                    "tam": result["summary"]["scores"]["Tam_Dao"],
                                    "tri": result["summary"]["scores"]["Tri_Dao"],
                                    "nghe": result["summary"]["scores"]["Su_Nghiep"],
                                    "error": "",
                                })
                            else:
                                rows.append({
                                    "file": name,
                                    "status": "error",
                                    "overall": "",
                                    "dominant": "",
                                    "sinh": "",
                                    "tam": "",
                                    "tri": "",
                                    "nghe": "",
                                    "error": result.get("error", "unknown"),
                                })

                        except Exception as inner:
                            rows.append({
                                "file": name,
                                "status": "error",
                                "overall": "",
                                "dominant": "",
                                "sinh": "",
                                "tam": "",
                                "tri": "",
                                "nghe": "",
                                "error": str(inner),
                            })

            else:
                img = Image.open(io.BytesIO(raw)).convert("RGB")
                result = engine.analyze_palm_image(img, f.filename or "image", parsed_settings)

                if result["ok"]:
                    rows.append({
                        "file": f.filename,
                        "status": "ok",
                        "overall": result["summary"]["overall"],
                        "dominant": result["summary"]["dominant_label"],
                        "sinh": result["summary"]["scores"]["Sinh_Dao"],
                        "tam": result["summary"]["scores"]["Tam_Dao"],
                        "tri": result["summary"]["scores"]["Tri_Dao"],
                        "nghe": result["summary"]["scores"]["Su_Nghiep"],
                        "error": "",
                    })
                else:
                    rows.append({
                        "file": f.filename,
                        "status": "error",
                        "overall": "",
                        "dominant": "",
                        "sinh": "",
                        "tam": "",
                        "tri": "",
                        "nghe": "",
                        "error": result.get("error", "unknown"),
                    })

        except Exception as e:
            rows.append({
                "file": f.filename,
                "status": "error",
                "overall": "",
                "dominant": "",
                "sinh": "",
                "tam": "",
                "tri": "",
                "nghe": "",
                "error": str(e),
            })

    return JSONResponse({"ok": True, "rows": rows})


def render_mystic_logo():
    return """
    <div class="logo-stage">
      <svg class="pv-logo" viewBox="0 0 760 760" aria-label="PalmVibe cyberpunk hand-eye logo" role="img">
        <defs>
          <linearGradient id="cyberLine" x1="90" y1="120" x2="650" y2="640">
            <stop offset="0%" stop-color="#73F7FF"/>
            <stop offset="42%" stop-color="#1BB9FF"/>
            <stop offset="72%" stop-color="#B12BFF"/>
            <stop offset="100%" stop-color="#FF2ECF"/>
          </linearGradient>

          <linearGradient id="glitchGrad" x1="80" y1="250" x2="680" y2="480">
            <stop offset="0%" stop-color="#00F5FF"/>
            <stop offset="38%" stop-color="#1664FF"/>
            <stop offset="68%" stop-color="#D728FF"/>
            <stop offset="100%" stop-color="#FF2BBF"/>
          </linearGradient>

          <radialGradient id="irisGrad" cx="50%" cy="48%" r="62%">
            <stop offset="0%" stop-color="#BFFFFF"/>
            <stop offset="28%" stop-color="#32E5FF"/>
            <stop offset="58%" stop-color="#1669FF"/>
            <stop offset="100%" stop-color="#050A18"/>
          </radialGradient>

          <filter id="glowStrong" x="-35%" y="-35%" width="170%" height="170%">
            <feGaussianBlur stdDeviation="6" result="b"/>
            <feColorMatrix in="b" type="matrix" values="0 0 0 0 0.20  0 0 0 0 0.95  0 0 0 0 1.00  0 0 0 .95 0" result="c"/>
            <feMerge>
              <feMergeNode in="c"/>
              <feMergeNode in="SourceGraphic"/>
            </feMerge>
          </filter>

          <filter id="glowPink" x="-45%" y="-45%" width="190%" height="190%">
            <feGaussianBlur stdDeviation="10" result="p"/>
            <feColorMatrix in="p" type="matrix" values="1 0 0 0 1  0 0 0 0 .08  0 0 0 0 .72  0 0 0 .85 0" result="pm"/>
            <feMerge>
              <feMergeNode in="pm"/>
              <feMergeNode in="SourceGraphic"/>
            </feMerge>
          </filter>

          <filter id="glowSoft" x="-45%" y="-45%" width="190%" height="190%">
            <feGaussianBlur stdDeviation="13" result="s"/>
            <feMerge>
              <feMergeNode in="s"/>
              <feMergeNode in="SourceGraphic"/>
            </feMerge>
          </filter>

          <clipPath id="eyeClipCyber">
            <path d="M286 422 C332 358 580 358 626 422 C580 486 332 486 286 422 Z"/>
          </clipPath>
        </defs>

        <rect x="0" y="0" width="760" height="760" fill="transparent"/>

        <g class="glitch-cloud">
          <g class="glitch-pack gp-a">
            <polygon class="pink" points="108,318 270,318 232,350 92,350"/>
            <polygon class="cyan" points="486,286 666,286 620,326 452,326"/>
            <polygon class="blue" points="122,442 300,442 252,476 84,476"/>
            <polygon class="pink" points="480,430 674,430 620,470 448,470"/>
            <rect class="cyan" x="68" y="374" width="214" height="12" rx="3"/>
            <rect class="pink" x="496" y="366" width="222" height="10" rx="3"/>
            <rect class="blue" x="154" y="260" width="118" height="8" rx="3"/>
            <rect class="cyan" x="498" y="520" width="142" height="8" rx="3"/>
          </g>

          <g class="glitch-pack gp-b">
            <rect class="pink" x="84" y="294" width="52" height="7" rx="2"/>
            <rect class="cyan" x="148" y="292" width="192" height="5" rx="2"/>
            <rect class="blue" x="430" y="250" width="206" height="5" rx="2"/>
            <rect class="pink" x="560" y="336" width="142" height="6" rx="2"/>
            <rect class="cyan" x="58" y="484" width="190" height="5" rx="2"/>
            <rect class="blue" x="476" y="492" width="218" height="5" rx="2"/>
            <rect class="pink" x="218" y="548" width="176" height="5" rx="2"/>
          </g>

          <g class="glitch-pack gp-c">
            <rect class="cyan" x="110" y="232" width="126" height="4" rx="2"/>
            <rect class="pink" x="288" y="226" width="84" height="4" rx="2"/>
            <rect class="blue" x="498" y="224" width="132" height="4" rx="2"/>
            <rect class="pink" x="34" y="408" width="180" height="4" rx="2"/>
            <rect class="cyan" x="556" y="404" width="170" height="4" rx="2"/>
            <rect class="blue" x="252" y="594" width="268" height="4" rx="2"/>
          </g>
        </g>

        <g class="logo-core">
          <g class="hand-symbol">
            <path class="hand-fill" d="
              M244 610
              C210 610 190 588 190 554
              L190 416
              C190 380 211 356 241 356
              C259 356 276 366 288 388
              L288 262
              C288 225 312 200 344 200
              C366 200 384 212 394 232
              L394 168
              C394 130 420 104 456 104
              C492 104 518 130 518 168
              L518 262
              C518 225 542 200 574 200
              C606 200 630 225 630 262
              L630 330
              C636 315 653 305 672 305
              C704 305 728 330 728 364
              L728 554
              C728 588 706 610 672 610
              Z"/>

            <path class="hand-line" d="
              M244 610
              C210 610 190 588 190 554
              L190 416
              C190 380 211 356 241 356
              C259 356 276 366 288 388
              L288 262
              C288 225 312 200 344 200
              C366 200 384 212 394 232
              L394 168
              C394 130 420 104 456 104
              C492 104 518 130 518 168
              L518 262
              C518 225 542 200 574 200
              C606 200 630 225 630 262
              L630 330
              C636 315 653 305 672 305
              C704 305 728 330 728 364
              L728 554
              C728 588 706 610 672 610
              Z"/>

            <path class="finger-line" d="M394 250 L394 340"/>
            <path class="finger-line" d="M518 262 L518 340"/>
            <path class="finger-line" d="M456 206 L456 340"/>
            <path class="center-line" d="M456 238 L456 332"/>
            <circle class="accent-dot" cx="456" cy="362" r="6"/>
            <circle class="accent-dot small" cx="456" cy="330" r="3"/>
            <circle class="accent-dot small" cx="456" cy="314" r="3"/>
            <circle class="accent-dot small" cx="456" cy="298" r="3"/>
          </g>

          <g class="eye-core">
            <path class="eye-outer" d="M286 422 C332 358 580 358 626 422 C580 486 332 486 286 422 Z"/>
            <path class="eye-white" d="M318 422 C362 382 550 382 594 422 C550 462 362 462 318 422 Z"/>

            <g clip-path="url(#eyeClipCyber)">
              <g class="iris-rays look-motion">
                <animateTransform
                  attributeName="transform"
                  type="translate"
                  values="0 0; -30 0; -30 0; 30 0; 30 0; 0 0"
                  keyTimes="0; .18; .35; .56; .73; 1"
                  dur="6.8s"
                  calcMode="spline"
                  keySplines=".42 0 .22 1; .42 0 .22 1; .42 0 .22 1; .42 0 .22 1; .42 0 .22 1"
                  repeatCount="indefinite"/>
                <circle class="iris" cx="456" cy="422" r="48"/>
                <circle class="pupil" cx="456" cy="422" r="23"/>
                <circle class="shine" cx="439" cy="401" r="10"/>
              </g>

              <rect class="lid-top" x="270" y="326" width="390" height="104" rx="56" transform="translate(0,-104)">
                <animateTransform
                  attributeName="transform"
                  type="translate"
                  values="0 -104; 0 -104; 0 22; 0 22; 0 -104"
                  keyTimes="0; .56; .70; .78; 1"
                  dur="6.8s"
                  calcMode="spline"
                  keySplines=".42 0 .22 1; .42 0 .22 1; .42 0 .22 1; .42 0 .22 1"
                  repeatCount="indefinite"/>
              </rect>

              <rect class="lid-bottom" x="270" y="414" width="390" height="104" rx="56" transform="translate(0,104)">
                <animateTransform
                  attributeName="transform"
                  type="translate"
                  values="0 104; 0 104; 0 -22; 0 -22; 0 104"
                  keyTimes="0; .56; .70; .78; 1"
                  dur="6.8s"
                  calcMode="spline"
                  keySplines=".42 0 .22 1; .42 0 .22 1; .42 0 .22 1; .42 0 .22 1"
                  repeatCount="indefinite"/>
              </rect>
            </g>

            <path class="eye-cut" d="M286 422 C332 358 580 358 626 422"/>
            <path class="eye-cut" d="M286 422 C332 486 580 486 626 422"/>
          </g>
        </g>
      </svg>
    </div>
    """


LOGO_HTML = render_mystic_logo()


HTML = """
<!DOCTYPE html>
<html lang="vi">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>PalmVibe</title>

  <style>
    :root{
      --bg:#020105;
      --panel:rgba(5,8,20,.86);
      --panel2:rgba(8,12,28,.76);
      --line:rgba(126,240,255,.18);
      --cyan:#67F7FF;
      --mag:#FF2ECF;
      --vio:#7C3CFF;
      --mint:#92FFA7;
      --text:#F8FAFC;
      --muted:#A8B3CF;
      --bad:#FF7B9C;
      --warn:#FFD166;
      --shadow:0 0 0 1px rgba(126,240,255,.10),0 28px 90px rgba(0,0,0,.42);
    }

    *{box-sizing:border-box}

    body{
      margin:0;
      min-height:100vh;
      color:var(--text);
      font-family:Inter,ui-sans-serif,system-ui,-apple-system,Segoe UI,Roboto,Arial,sans-serif;
      background:
        radial-gradient(circle at 12% 7%, rgba(126,240,255,.18), transparent 30%),
        radial-gradient(circle at 90% 15%, rgba(255,97,216,.18), transparent 28%),
        radial-gradient(circle at 52% 88%, rgba(140,123,255,.16), transparent 28%),
        linear-gradient(180deg,#02040A 0%,#050713 100%);
      overflow-x:hidden;
    }

    body::before{
      content:"";
      position:fixed;
      inset:0;
      z-index:-1;
      background-image:
        linear-gradient(rgba(255,255,255,.035) 1px, transparent 1px),
        linear-gradient(90deg, rgba(255,255,255,.035) 1px, transparent 1px);
      background-size:46px 46px;
      mask-image:linear-gradient(to bottom, rgba(255,255,255,.85), rgba(255,255,255,.18));
    }

    .wrap{
      width:min(1420px,94vw);
      margin:0 auto;
      padding:28px 0 80px;
    }

    .hero{
      display:grid;
      grid-template-columns:1.12fr .88fr;
      gap:26px;
      align-items:center;
      min-height:470px;
      padding:34px;
      border-radius:34px;
      background:linear-gradient(135deg,rgba(126,240,255,.10),rgba(255,97,216,.07));
      border:1px solid rgba(126,240,255,.18);
      box-shadow:var(--shadow);
      overflow:hidden;
      position:relative;
    }

    .hero h1{
      font-size:76px;
      line-height:.94;
      margin:0 0 18px;
      letter-spacing:-.055em;
      font-weight:950;
    }

    .gradient{
      background:linear-gradient(90deg,var(--cyan),#E9FCFF 34%,var(--mag) 92%);
      -webkit-background-clip:text;
      background-clip:text;
      color:transparent;
    }

    .hero p{
      color:var(--muted);
      font-size:20px;
      line-height:1.65;
      max-width:900px;
      margin:0;
    }

    .chips{
      display:flex;
      flex-wrap:wrap;
      gap:12px;
      margin-top:22px;
    }

    .chip{
      padding:11px 16px;
      border-radius:999px;
      background:rgba(9,17,35,.75);
      border:1px solid rgba(126,240,255,.22);
      font-weight:850;
      color:#EAFBFF;
    }

    .status-grid{
      display:grid;
      grid-template-columns:repeat(3,1fr);
      gap:14px;
      margin-top:24px;
    }

    .status-card,
    .main-section,
    .summary-card,
    .img-card,
    .roi-card,
    .viz-card,
    .batch-card,
    .interpret-card,
    .post-section{
      background:var(--panel);
      border:1px solid rgba(126,240,255,.15);
      border-radius:28px;
      box-shadow:var(--shadow);
    }

    .status-card{
      padding:18px;
    }

    .status-card .k{
      color:var(--muted);
      font-size:14px;
    }

    .status-card .v{
      margin-top:4px;
      font-size:25px;
      font-weight:950;
    }

    .ok{color:var(--mint)}
    .bad{color:var(--bad)}

    .logo-stage{
      display:grid;
      place-items:center;
      width:100%;
      min-height:420px;
      isolation:isolate;
    }

    .pv-logo{
      width:min(430px,86%);
      height:auto;
      overflow:visible;
      filter:drop-shadow(0 0 34px rgba(255,46,207,.16));
    }

    .logo-core{
      transform-box:fill-box;
      transform-origin:center;
      transform:translate(-70px,4px) scale(.86);
    }

    .glitch-cloud{
      opacity:.98;
      filter:url(#glowSoft);
    }

    .glitch-pack .cyan{fill:#00F5FF}
    .glitch-pack .pink{fill:#FF2ECF}
    .glitch-pack .blue{fill:#155BFF}

    .gp-a{animation:cyberGlitchA 2.6s steps(1,end) infinite; opacity:.92}
    .gp-b{animation:cyberGlitchB 1.9s steps(1,end) infinite; opacity:.86}
    .gp-c{animation:cyberGlitchC 1.45s steps(1,end) infinite; opacity:.74}

    .hand-fill{
      fill:rgba(2,5,13,.985);
    }

    .hand-line{
      fill:none;
      stroke:#75F7FF;
      stroke-width:12;
      stroke-linejoin:round;
      stroke-linecap:round;
      filter:url(#glowStrong);
    }

    .finger-line{
      fill:none;
      stroke:#75F7FF;
      stroke-width:8;
      stroke-linecap:round;
      opacity:.95;
      filter:url(#glowStrong);
    }

    .center-line{
      fill:none;
      stroke:url(#cyberLine);
      stroke-width:5;
      stroke-linecap:round;
      filter:url(#glowPink);
    }

    .accent-dot{
      fill:#3BDFFF;
      filter:url(#glowStrong);
    }

    .accent-dot.small{
      fill:#FF2ECF;
      opacity:.95;
    }

    .eye-core{
      filter:drop-shadow(0 0 22px rgba(0,245,255,.72));
    }

    .eye-outer{
      fill:rgba(2,5,13,.96);
      stroke:#F4FDFF;
      stroke-width:9;
      stroke-linejoin:round;
      filter:url(#glowStrong);
    }

    .eye-white{
      fill:#F6FBFF;
    }

    .iris{
      fill:url(#irisGrad);
      filter:url(#glowStrong);
      transform-origin:456px 422px;
      animation:irisPulse 4s ease-in-out infinite;
    }

    .iris-rays::before{content:""}

    .look-motion{
      transform-box:fill-box;
      transform-origin:center;
      will-change:transform;
    }

    .lid-top,
    .lid-bottom{
      fill:#02050D;
      opacity:.99;
      filter:url(#glowStrong);
    }

    .pupil{
      fill:#020711;
    }

    .shine{
      fill:white;
      opacity:.96;
    }

    .eye-cut{
      fill:none;
      stroke:#F4FDFF;
      stroke-width:5;
      stroke-linecap:round;
      opacity:.85;
    }

    @keyframes cyberGlitchA{
      0%,100%{transform:translate(0,0); opacity:.88}
      12%{transform:translate(-18px,2px); opacity:1}
      13%{transform:translate(14px,-2px); opacity:.96}
      14%{transform:translate(0,0); opacity:.9}
      48%{transform:translate(10px,0); opacity:.98}
      49%{transform:translate(-8px,0); opacity:.84}
      50%{transform:translate(0,0)}
      76%{transform:translate(-12px,1px); opacity:1}
      77%{transform:translate(0,0)}
    }

    @keyframes cyberGlitchB{
      0%,100%{transform:translate(0,0); opacity:.7}
      18%{transform:translate(20px,-1px); opacity:1}
      19%{transform:translate(-16px,1px); opacity:.78}
      20%{transform:translate(0,0)}
      57%{transform:translate(-12px,0); opacity:.95}
      58%{transform:translate(12px,0)}
      59%{transform:translate(0,0)}
    }

    @keyframes cyberGlitchC{
      0%,100%{transform:translate(0,0); opacity:.56}
      31%{transform:translate(-24px,0); opacity:.94}
      32%{transform:translate(22px,0); opacity:.75}
      33%{transform:translate(0,0)}
      70%{transform:translate(14px,0); opacity:.9}
      71%{transform:translate(-10px,0)}
      72%{transform:translate(0,0)}
    }

    @keyframes irisPulse{
      0%,100%{transform:scale(1); filter:url(#glowStrong)}
      50%{transform:scale(1.035); filter:url(#glowPink)}
    }

    .main-section{
      margin-top:30px;
      padding:24px;
    }

    .tabs{
      display:flex;
      flex-wrap:wrap;
      gap:14px;
      margin-bottom:22px;
    }

    button{
      border:none;
      border-radius:18px;
      padding:14px 22px;
      color:white;
      background:linear-gradient(90deg,rgba(126,240,255,.18),rgba(255,97,216,.20));
      border:1px solid rgba(126,240,255,.22);
      font-weight:900;
      cursor:pointer;
      box-shadow:0 12px 28px rgba(0,0,0,.22);
    }

    button:hover{
      box-shadow:0 0 28px rgba(126,240,255,.18),0 12px 28px rgba(0,0,0,.22);
    }

    .tab.active{
      border-color:rgba(126,240,255,.55);
      background:linear-gradient(90deg,rgba(126,240,255,.22),rgba(255,97,216,.24));
    }

    .panel{
      display:none;
    }

    .panel.active{
      display:block;
    }

    .panel h2{
      margin:4px 0 8px;
      font-size:32px;
    }

    .panel p{
      margin-top:0;
      color:var(--muted);
      font-size:17px;
      line-height:1.65;
    }

    .upload-box{
      padding:18px;
      border:1px dashed rgba(126,240,255,.26);
      border-radius:24px;
      background:rgba(8,14,30,.72);
    }

    input[type=file]{
      width:100%;
      padding:16px;
      color:white;
      background:#091124;
      border:1px solid rgba(126,240,255,.18);
      border-radius:18px;
    }

    .camera-box{
      position:relative;
      min-height:560px;
      overflow:hidden;
      border-radius:28px;
      background:#02040A;
      border:1px solid rgba(126,240,255,.22);
      display:grid;
      place-items:center;
    }

    video,
    .camera-preview{
      width:100%;
      height:100%;
      object-fit:cover;
      display:block;
      min-height:560px;
      background:#02040A;
      filter:brightness(.96) contrast(1.02) saturate(1.02);
    }

    .guide{
      position:absolute;
      inset:0;
      pointer-events:none;
      background:rgba(0,0,0,.28);
    }

    .guide::before{
      content:"";
      position:absolute;
      left:50%;
      top:50%;
      transform:translate(-50%,-50%);
      width:58%;
      height:76%;
      border:4px solid rgba(126,240,255,.88);
      border-radius:30px;
      box-shadow:0 0 34px rgba(126,240,255,.26), inset 0 0 30px rgba(126,240,255,.06);
    }

    .guide::after{
      content:"";
      position:absolute;
      left:50%;
      top:50%;
      width:52%;
      height:2px;
      transform:translate(-50%,-50%);
      background:linear-gradient(90deg,transparent,var(--cyan),var(--mag),transparent);
      box-shadow:0 0 16px rgba(255,97,216,.5);
      animation:scanLine 3.4s ease-in-out infinite;
    }

    .guide-label{
      position:absolute;
      left:50%;
      top:12px;
      transform:translateX(-50%);
      padding:10px 18px;
      border-radius:999px;
      background:rgba(7,14,29,.88);
      border:1px solid rgba(126,240,255,.22);
      font-weight:900;
      z-index:5;
    }

    .shutter-btn{
      position:absolute;
      left:50%;
      bottom:22px;
      transform:translateX(-50%);
      width:84px;
      height:84px;
      border-radius:50%;
      background:rgba(255,255,255,.18);
      border:2px solid rgba(255,255,255,.72);
      display:grid;
      place-items:center;
      z-index:5;
      padding:0;
      box-shadow:0 12px 30px rgba(0,0,0,.26);
      backdrop-filter:blur(6px);
    }

    .shutter-btn::before{
      content:"";
      width:60px;
      height:60px;
      border-radius:50%;
      background:white;
      box-shadow:0 0 18px rgba(255,255,255,.26);
    }

    .shutter-btn:hover{
      transform:translateX(-50%) scale(1.03);
    }

    @keyframes scanLine{
      0%,100%{transform:translate(-50%,-200px)}
      50%{transform:translate(-50%,200px)}
    }

    .alert,
    .warning{
      margin-top:18px;
      padding:18px 20px;
      border-radius:20px;
      font-size:17px;
    }

    .alert{
      border:1px solid rgba(255,123,156,.35);
      background:linear-gradient(90deg,rgba(90,14,36,.42),rgba(72,18,62,.34));
      color:#FFEAF0;
    }

    .warning{
      border:1px solid rgba(255,209,102,.38);
      background:linear-gradient(90deg,rgba(90,70,18,.28),rgba(82,50,18,.24));
      color:#FFF5D4;
    }

    .results{
      display:none;
      margin-top:30px;
    }

    .summary-grid{
      display:grid;
      grid-template-columns:1fr .72fr .72fr;
      gap:18px;
      margin-bottom:20px;
    }

    .summary-card{
      padding:24px;
    }

    .summary-card .k{
      text-transform:uppercase;
      letter-spacing:.08em;
      font-size:14px;
      color:var(--muted);
      font-weight:900;
    }

    .summary-card .big{
      font-size:74px;
      line-height:1;
      color:var(--mint);
      font-weight:950;
      margin:10px 0;
    }

    .summary-card .mid{
      font-size:23px;
      font-weight:950;
    }

    .img-grid{
      display:grid;
      grid-template-columns:repeat(2,1fr);
      gap:18px;
      margin:20px 0;
    }

    .img-card{
      padding:18px;
    }

    .img-card h3{
      margin:0 0 14px;
      font-size:23px;
    }

    .img-card img,
    .roi-card img{
      width:100%;
      border-radius:20px;
      border:1px solid rgba(126,240,255,.16);
      display:block;
    }

    .roi-grid{
      display:grid;
      grid-template-columns:repeat(4,1fr);
      gap:18px;
      margin:20px 0;
    }

    .roi-card{
      padding:18px;
      overflow:hidden;
    }

    .roi-card h3{
      margin:16px 0 8px;
      font-size:24px;
    }

    .desc{
      color:var(--muted);
      min-height:46px;
      line-height:1.5;
    }

    .badge{
      display:inline-flex;
      padding:8px 14px;
      border-radius:999px;
      border:1px solid rgba(126,240,255,.28);
      background:rgba(7,19,36,.72);
      font-weight:900;
      margin:12px 0;
    }

    .score{
      font-size:66px;
      line-height:1;
      font-weight:950;
      color:var(--mint);
      margin:6px 0 10px;
    }

    .bar{
      height:14px;
      border-radius:999px;
      background:rgba(255,255,255,.10);
      overflow:hidden;
      margin:8px 0 14px;
    }

    .bar span{
      display:block;
      height:100%;
      border-radius:999px;
      background:linear-gradient(90deg,var(--cyan),var(--mag));
    }

    .meta{
      display:grid;
      grid-template-columns:1fr 1fr;
      gap:8px;
      color:#DDE8FF;
      font-size:15px;
      line-height:1.4;
    }

    .model-note{
      color:#A8B3CF;
      font-size:13px;
      line-height:1.5;
      margin-top:12px;
    }

    .viz-grid{
      display:grid;
      grid-template-columns:1.15fr .85fr;
      gap:18px;
      margin-top:20px;
    }

    .viz-card{
      padding:24px;
    }

    .viz-card h3{
      margin:0 0 18px;
      font-size:30px;
    }

    table{
      width:100%;
      border-collapse:separate;
      border-spacing:10px;
    }

    th{
      text-align:left;
      color:#F4FAFF;
      font-size:16px;
    }

    td{
      padding:16px 12px;
      border-radius:18px;
      background:rgba(14,22,44,.86);
      border:1px solid rgba(126,240,255,.12);
      font-weight:900;
      text-align:center;
    }

    td.rowhead{
      background:transparent;
      border:none;
      text-align:left;
      padding-left:0;
    }

    .interpret-card{
      margin-top:20px;
      padding:24px;
    }

    .interpret-tabs{
      display:flex;
      flex-wrap:wrap;
      gap:10px;
      margin-bottom:16px;
    }

    .interpret-tab{
      padding:10px 14px;
      border-radius:999px;
      background:rgba(10,18,38,.8);
      border:1px solid rgba(126,240,255,.15);
      cursor:pointer;
      font-weight:850;
    }

    .interpret-tab.active{
      border-color:rgba(126,240,255,.55);
      background:rgba(126,240,255,.14);
    }

    .interpret-content{
      padding:18px;
      border-radius:22px;
      background:rgba(8,14,30,.72);
      border:1px solid rgba(126,240,255,.12);
      color:#EEF6FF;
      line-height:1.7;
      font-size:18px;
    }

    .interpret-content p{
      margin:0 0 14px;
    }

    .post-section{
      margin-top:20px;
      padding:24px;
    }

    .post-section h3{
      font-size:30px;
      margin:0 0 8px;
    }

    .post-section > p{
      color:var(--muted);
      margin:0 0 18px;
      font-size:17px;
      line-height:1.6;
    }

    .post-grid{
      display:grid;
      grid-template-columns:repeat(4,1fr);
      gap:16px;
    }

    .post-card{
      display:block;
      text-decoration:none;
      color:var(--text);
      background:rgba(8,14,30,.72);
      border:1px solid rgba(126,240,255,.16);
      border-radius:24px;
      overflow:hidden;
      transition:.22s ease;
      min-height:100%;
    }

    .post-card:hover{
      transform:translateY(-4px);
      border-color:rgba(126,240,255,.5);
      box-shadow:0 0 36px rgba(126,240,255,.12);
    }

    .post-card img{
      width:100%;
      height:150px;
      object-fit:cover;
      display:block;
      filter:saturate(1.08) contrast(1.02);
    }

    .post-body{
      padding:16px;
    }

    .post-tag{
      display:inline-flex;
      padding:6px 10px;
      border-radius:999px;
      color:#EAFBFF;
      font-size:12px;
      font-weight:900;
      background:linear-gradient(90deg,rgba(126,240,255,.16),rgba(255,97,216,.18));
      border:1px solid rgba(126,240,255,.16);
      margin-bottom:10px;
    }

    .post-title{
      font-weight:950;
      font-size:18px;
      line-height:1.25;
      margin-bottom:8px;
    }

    .post-desc{
      color:var(--muted);
      line-height:1.55;
      font-size:14px;
    }

    .batch-card{
      margin-top:22px;
      padding:18px;
      overflow:auto;
    }

    .batch-card table{
      min-width:850px;
      border-spacing:8px;
    }

    .loading{
      position:fixed;
      inset:0;
      z-index:999;
      display:none;
      place-items:center;
      background:rgba(2,4,10,.74);
      backdrop-filter:blur(14px);
    }

    .loading.active{
      display:grid;
    }

    .loading-card{
      width:min(560px,90vw);
      padding:30px;
      text-align:center;
      border-radius:34px;
      background:rgba(5,10,22,.96);
      border:1px solid rgba(126,240,255,.18);
      box-shadow:0 28px 100px rgba(0,0,0,.48);
    }

    .loading-card .pv-logo{
      width:290px;
    }

    .loading-card .logo-stage{
      min-height:300px;
    }

    .loading-title{
      font-size:38px;
      font-weight:950;
      margin-top:10px;
    }

    .loading-sub{
      font-size:17px;
      color:var(--muted);
      margin-top:10px;
    }

    footer{
      text-align:center;
      color:var(--muted);
      margin-top:36px;
    }

    @media(max-width:1120px){
      .hero,
      .summary-grid,
      .img-grid,
      .roi-grid,
      .viz-grid,
      .post-grid{
        grid-template-columns:1fr;
      }

      .hero h1{
        font-size:52px;
      }

      .camera-box{
        min-height:500px;
      }

      video,
      .camera-preview{
        min-height:500px;
      }
    }
  </style>
</head>

<body>
  <div class="wrap">
    <section class="hero">
      <div>
        <h1><span class="gradient">PalmVibe</span><br/>Cyber Palm Scanner</h1>
        <p>
          A cyber-style AI demo that scans palm images, highlights key palm regions, and turns visual signals into an interactive self-discovery dashboard.
        </p>

        <div class="chips">
          <span class="chip">Palm Image AI</span>
          <span class="chip">Cyber Dashboard</span>
          <span class="chip">Camera + Upload</span>
          <span class="chip">Learning Project</span>
        </div>

        <div class="status-grid">
          <div class="status-card"><div class="k">AI Model</div><div id="stModel" class="v">...</div></div>
          <div class="status-card"><div class="k">Hand Scanner</div><div id="stMP" class="v">...</div></div>
          <div class="status-card"><div class="k">Label Map</div><div id="stClass" class="v">...</div></div>
        </div>
      </div>

      <div>
        __LOGO__
      </div>
    </section>

    <section class="main-section">
      <div class="tabs">
        <button class="tab active" data-tab="upload">Upload Single Image</button>
        <button class="tab" data-tab="camera">Camera Scan</button>
        <button class="tab" data-tab="batch">Batch / Zip</button>
      </div>

      <div id="panel-upload" class="panel active">
        <h2>Upload Single Image</h2>
        <p>Upload ảnh JPG/PNG/WEBP. App sẽ quét lòng bàn tay và dựng bảng kết quả tương tác.</p>
        <div class="upload-box">
          <input type="file" id="uploadFile" accept="image/*"/>
          <div style="height:16px"></div>
          <button id="btnUpload">Analyze image </button>
        </div>
        <div id="uploadAlert"></div>
      </div>

      <div id="panel-camera" class="panel">
        <h2>Camera Scan</h2>
        <p>Camera chạy qua HTTPS ngrok. Ảnh chụp sẽ được đưa vào đúng cùng pipeline như lúc upload.</p>

        <div class="camera-box">
          <video id="video" playsinline autoplay muted></video>
          <canvas id="cameraCanvas" class="camera-preview" style="display:none"></canvas>
          <div class="guide"></div>
          <div class="guide-label">Đặt lòng bàn tay vào khung</div>
          <button id="btnShutter" class="shutter-btn" title="Take photo"></button>
        </div>

        <div style="height:16px"></div>
        <button id="btnStartCamera">Start Camera</button>
        <button id="btnCapture">Take Photo & Scan</button>
        <div id="cameraAlert"></div>
      </div>

      <div id="panel-batch" class="panel">
        <h2>Batch / Zip</h2>
        <p>Tải nhiều ảnh hoặc file zip. Mỗi ảnh sẽ được quét độc lập và tổng hợp thành bảng kết quả.</p>
        <div class="upload-box">
          <input type="file" id="batchFiles" accept="image/*,.zip" multiple/>
          <div style="height:16px"></div>
          <button id="btnBatch">Analyze batch</button>
          <button id="btnCsv" style="display:none">Tải CSV</button>
        </div>
        <div id="batchAlert"></div>
        <div id="batchResults"></div>
      </div>
    </section>

    <section id="results" class="results"></section>
    <footer>PalmVibe • AI palm image • Project</footer>
  </div>

  <div id="loading" class="loading">
    <div class="loading-card">
      __LOGO__
      <div class="loading-title">Loading PalmVibe scan...</div>
      <div class="loading-sub">Đang quét ảnh lòng bàn tay • dựng bảng kết quả</div>
    </div>
  </div>

  <script>
    const initialStatus = __STATUS__;

    let currentStream = null;
    let lastBatchRows = [];

    function $(id){ return document.getElementById(id); }

    function setStatus(st){
      $("stModel").textContent = st.model_ok ? "Ready" : "Missing";
      $("stMP").textContent = st.mediapipe_ok ? "Ready" : "Missing";
      $("stClass").textContent = st.class_ok ? "Loaded" : "Fallback";

      $("stModel").className = "v " + (st.model_ok ? "ok" : "bad");
      $("stMP").className = "v " + (st.mediapipe_ok ? "ok" : "bad");
      $("stClass").className = "v ok";
    }

    setStatus(initialStatus);
    fetch("/api/status").then(r => r.json()).then(setStatus).catch(() => {});

    document.querySelectorAll(".tab").forEach(btn => {
      btn.addEventListener("click", () => {
        document.querySelectorAll(".tab").forEach(b => b.classList.remove("active"));
        btn.classList.add("active");

        document.querySelectorAll(".panel").forEach(p => p.classList.remove("active"));
        $("panel-" + btn.dataset.tab).classList.add("active");
      });
    });

    function showLoading(v){
      $("loading").classList.toggle("active", v);
    }

    function alertBox(id, msg, type="alert"){
      $(id).innerHTML = msg ? `<div class="${type}">${msg}</div>` : "";
    }

    async function analyzeBlob(blob, filename){
      showLoading(true);
      $("results").style.display = "none";

      try{
        const form = new FormData();
        form.append("file", blob, filename);
        form.append("settings", JSON.stringify({}));

        const res = await fetch("/api/analyze", {
          method: "POST",
          body: form
        });

        const data = await res.json();
        showLoading(false);

        if (!data.ok){
          renderError(data.error || "Analysis failed.");
          return;
        }

        renderResults(data);

        setTimeout(() => {
          $("results").scrollIntoView({behavior:"smooth", block:"start"});
        }, 100);

      }catch(err){
        showLoading(false);
        renderError("API error: " + err.message);
      }
    }

    function renderError(msg){
      $("results").style.display = "block";
      $("results").innerHTML = `<div class="alert">${msg}</div>`;
    }

    $("btnUpload").addEventListener("click", async () => {
      const f = $("uploadFile").files[0];

      if (!f){
        alertBox("uploadAlert", "Vui lòng chọn một ảnh.");
        return;
      }

      alertBox("uploadAlert", "");
      await analyzeBlob(f, f.name || "upload.jpg");
    });

    async function startCamera(){
      alertBox("cameraAlert", "");

      try{
        if (currentStream){
          currentStream.getTracks().forEach(t => t.stop());
          currentStream = null;
        }

        const video = $("video");
        const canvas = $("cameraCanvas");

        canvas.style.display = "none";
        video.style.display = "block";

        let stream = null;

        try{
          stream = await navigator.mediaDevices.getUserMedia({
            video: {
              width:{ideal:1280},
              height:{ideal:720},
              facingMode:"user"
            },
            audio:false
          });
        }catch(e1){
          stream = await navigator.mediaDevices.getUserMedia({
            video:true,
            audio:false
          });
        }

        currentStream = stream;
        video.srcObject = stream;
        video.setAttribute("playsinline", "true");
        video.muted = true;
        await video.play();

      }catch(err){
        alertBox("cameraAlert", "Camera chưa được cấp quyền hoặc browser đang chặn nguồn video. " + err.message);
      }
    }

    $("btnStartCamera").addEventListener("click", startCamera);

    function normalizeCanvasExposure(canvas){
      const ctx = canvas.getContext("2d");
      const img = ctx.getImageData(0, 0, canvas.width, canvas.height);
      const d = img.data;
      let sum = 0;

      for (let i = 0; i < d.length; i += 4){
        sum += 0.2126 * d[i] + 0.7152 * d[i + 1] + 0.0722 * d[i + 2];
      }

      const avg = sum / (d.length / 4) / 255;
      let gain = 1.0;
      let contrast = 1.02;

      // Camera laptop hay bị cháy sáng: chỉ kéo highlight xuống nhẹ, không phủ filter nặng.
      if (avg > 0.82){
        gain = 0.72;
        contrast = 1.08;
      }else if (avg > 0.74){
        gain = 0.80;
        contrast = 1.06;
      }else if (avg > 0.66){
        gain = 0.88;
        contrast = 1.04;
      }else if (avg < 0.34){
        gain = 1.10;
        contrast = 1.04;
      }

      for (let i = 0; i < d.length; i += 4){
        for (let c = 0; c < 3; c++){
          let v = d[i + c] / 255;
          v = ((v - 0.5) * contrast + 0.5) * gain;
          d[i + c] = Math.max(0, Math.min(255, Math.round(v * 255)));
        }
      }

      ctx.putImageData(img, 0, 0);
      return {avg, gain, contrast};
    }

    function captureGuideBlob(){
      return new Promise((resolve, reject) => {
        const video = $("video");

        if (!video.videoWidth || !video.videoHeight){
          reject(new Error("Camera chưa sẵn sàng."));
          return;
        }

        const vw = video.videoWidth;
        const vh = video.videoHeight;

        const sx = vw * 0.21;
        const sy = vh * 0.12;
        const sw = vw * 0.58;
        const sh = vh * 0.76;

        const canvas = document.createElement("canvas");
        canvas.width = Math.floor(sw);
        canvas.height = Math.floor(sh);

        const ctx = canvas.getContext("2d");
        ctx.filter = "none";
        ctx.drawImage(video, sx, sy, sw, sh, 0, 0, canvas.width, canvas.height);
        normalizeCanvasExposure(canvas);

        const preview = $("cameraCanvas");
        preview.width = canvas.width;
        preview.height = canvas.height;
        const pctx = preview.getContext("2d");
        pctx.filter = "none";
        pctx.drawImage(canvas, 0, 0);

        preview.style.display = "block";
        video.style.display = "none";

        canvas.toBlob(blob => {
          if (!blob){
            reject(new Error("Không thể chụp frame từ camera."));
            return;
          }
          resolve(blob);
        }, "image/jpeg", 0.96);
      });
    }

    async function captureAndAnalyze(){
      try{
        if (!currentStream){
          await startCamera();
        }

        const blob = await captureGuideBlob();
        await analyzeBlob(blob, "camera_capture.jpg");

      }catch(err){
        alertBox("cameraAlert", err.message);
      }
    }

    $("btnCapture").addEventListener("click", captureAndAnalyze);
    $("btnShutter").addEventListener("click", captureAndAnalyze);

    $("btnBatch").addEventListener("click", async () => {
      const files = Array.from($("batchFiles").files);

      if (!files.length){
        alertBox("batchAlert", "Vui lòng chọn ảnh hoặc file zip.");
        return;
      }

      showLoading(true);
      alertBox("batchAlert", "");

      try{
        const form = new FormData();

        files.forEach(f => {
          form.append("files", f, f.name);
        });

        form.append("settings", JSON.stringify({}));

        const res = await fetch("/api/analyze_batch", {
          method: "POST",
          body: form
        });

        const data = await res.json();
        showLoading(false);

        if (!data.ok){
          alertBox("batchAlert", data.error || "Batch failed.");
          return;
        }

        lastBatchRows = data.rows || [];
        renderBatch(lastBatchRows);
        $("btnCsv").style.display = "inline-block";

      }catch(err){
        showLoading(false);
        alertBox("batchAlert", "Lỗi API batch: " + err.message);
      }
    });

    $("btnCsv").addEventListener("click", () => {
      if (!lastBatchRows.length){
        return;
      }

      const headers = Object.keys(lastBatchRows[0]);
      const lines = [headers.join(",")];

      lastBatchRows.forEach(row => {
        lines.push(headers.map(h => `"${String(row[h] ?? "").replaceAll('"','""')}"`).join(","));
      });

      const blob = new Blob([lines.join("\\n")], {type:"text/csv;charset=utf-8"});
      const url = URL.createObjectURL(blob);

      const a = document.createElement("a");
      a.href = url;
      a.download = "palmvibe_batch_results.csv";
      document.body.appendChild(a);
      a.click();
      a.remove();

      URL.revokeObjectURL(url);
    });

    function renderBatch(rows){
      const html = `
        <div class="batch-card">
          <h3>Kết quả batch</h3>
          <table>
            <thead>
              <tr>
                <th>File</th>
                <th>Trạng thái</th>
                <th>Tổng điểm</th>
                <th>Vùng nổi bật</th>
                <th>Sinh Đạo</th>
                <th>Tâm Đạo</th>
                <th>Trí Đạo</th>
                <th>Sự Nghiệp</th>
                <th>Lỗi</th>
              </tr>
            </thead>
            <tbody>
              ${rows.map(r => `
                <tr>
                  <td>${r.file}</td>
                  <td>${r.status}</td>
                  <td>${r.overall}</td>
                  <td>${r.dominant}</td>
                  <td>${r.sinh}</td>
                  <td>${r.tam}</td>
                  <td>${r.tri}</td>
                  <td>${r.nghe}</td>
                  <td>${r.error}</td>
                </tr>
              `).join("")}
            </tbody>
          </table>
        </div>
      `;

      $("batchResults").innerHTML = html;
    }

    function heatmapHTML(h){
      let html = `
        <div class="viz-card">
          <h3>Ma trận xác suất AI</h3>
          <table>
            <thead>
              <tr>
                <th>Vùng</th>
                ${h.cols.map(c => `<th>${c}</th>`).join("")}
              </tr>
            </thead>
            <tbody>
      `;

      h.rows.forEach((r, i) => {
        html += `<tr><td class="rowhead">${r}</td>`;

        h.data[i].forEach(v => {
          const a = Math.min(0.16 + v / 125, 0.92);
          html += `<td style="background:linear-gradient(90deg,rgba(126,240,255,${a}),rgba(255,97,216,${a}))">${v.toFixed(1)}%</td>`;
        });

        html += `</tr>`;
      });

      html += `</tbody></table></div>`;
      return html;
    }

    function radarHTML(r){
      const cx = 220;
      const cy = 220;
      const R = 150;
      const n = r.labels.length;

      function pt(i, ratio){
        const angle = -Math.PI / 2 + i * 2 * Math.PI / n;
        return {
          x: cx + Math.cos(angle) * R * ratio,
          y: cy + Math.sin(angle) * R * ratio
        };
      }

      let grid = "";

      [0.25, 0.5, 0.75, 1].forEach(level => {
        let pts = [];
        for (let i = 0; i < n; i++){
          const p = pt(i, level);
          pts.push(`${p.x},${p.y}`);
        }
        grid += `<polygon points="${pts.join(" ")}" fill="none" stroke="rgba(255,255,255,.12)" stroke-width="1"/>`;
      });

      let lines = "";
      let labels = "";

      for (let i = 0; i < n; i++){
        const p = pt(i, 1);
        lines += `<line x1="${cx}" y1="${cy}" x2="${p.x}" y2="${p.y}" stroke="rgba(255,255,255,.12)" stroke-width="1"/>`;

        const t = pt(i, 1.18);
        labels += `<text x="${t.x}" y="${t.y}" fill="#EAF2FF" font-size="16" text-anchor="middle" dominant-baseline="middle">${r.labels[i]}</text>`;
      }

      let poly = [];

      for (let i = 0; i < n; i++){
        const p = pt(i, r.values[i] / 10);
        poly.push(`${p.x},${p.y}`);
      }

      return `
        <div class="viz-card">
          <h3>Radar tổng hợp</h3>
          <svg viewBox="0 0 440 440" width="100%" style="max-width:460px;display:block;margin:auto">
            ${grid}
            ${lines}
            <polygon points="${poly.join(" ")}" fill="rgba(126,240,255,.24)" stroke="#A7F7FF" stroke-width="4"/>
            <circle cx="${cx}" cy="${cy}" r="6" fill="#FF61D8"/>
            ${labels}
          </svg>
        </div>
      `;
    }

    function renderInterpretation(data){
      const keys = Object.keys(data.interpretation);

      return `
        <div class="interpret-card">
          <h3 style="font-size:30px;margin:0 0 18px">Kết quả phân tích</h3>
          <div class="interpret-tabs">
            ${keys.map((k, i) => `<div class="interpret-tab ${i === 0 ? "active" : ""}" data-i="${i}">${k}</div>`).join("")}
          </div>
          <div class="interpret-content" id="interpretContent">
            ${data.interpretation[keys[0]].map(x => `<p>${x}</p>`).join("")}
          </div>
        </div>
      `;
    }

    function renderRelatedPosts(data){
      const posts = data.related_posts || [];

      return `
        <div class="post-section">
          <h3>Khám phá thêm</h3>
          <p>Những bài đọc này giúp người dùng hiểu sâu hơn về tính cách, cảm xúc, định hướng và cách phát triển bản thân sau khi xem kết quả.</p>

          <div class="post-grid">
            ${posts.map(p => `
              <a class="post-card" href="${p.url}" target="_blank" rel="noopener noreferrer">
                <img src="${p.img}" alt="${p.title}" loading="lazy">
                <div class="post-body">
                  <div class="post-tag">${p.tag}</div>
                  <div class="post-title">${p.title}</div>
                  <div class="post-desc">${p.desc}</div>
                </div>
              </a>
            `).join("")}
          </div>
        </div>
      `;
    }

    function activateInterpretation(data){
      const keys = Object.keys(data.interpretation);

      document.querySelectorAll(".interpret-tab").forEach(tab => {
        tab.addEventListener("click", () => {
          document.querySelectorAll(".interpret-tab").forEach(t => t.classList.remove("active"));
          tab.classList.add("active");

          const i = parseInt(tab.dataset.i);
          $("interpretContent").innerHTML = data.interpretation[keys[i]].map(x => `<p>${x}</p>`).join("");
        });
      });
    }

    function renderResults(data){
      const cards = data.rois.map(r => `
        <div class="roi-card">
          <img src="${r.crop}" alt="${r.label}">
          <h3>${r.label}</h3>
          <div class="desc">${r.description}</div>

          <div class="badge">${r.level}</div>

          <div class="score">${r.score}<span style="font-size:.58em;color:#B7C4E1">/10</span></div>
          <div class="bar"><span style="width:${r.score * 10}%"></span></div>

          <div class="meta">
            <div>Ảnh: <b>${r.image_score}</b></div>
            <div>AI: <b>${r.ai_score}</b></div>
            <div>Độ tin ROI: <b>${r.roi_confidence}%</b></div>
            <div>Độ chắc: <b>${r.certainty}%</b></div>
          </div>

          <div class="model-note">
            Tín hiệu AI nghiêng về <b>${r.raw_model_signal}</b>. Điểm cuối đã được cân bằng với vùng lòng bàn tay phát hiện được.
          </div>
        </div>
      `).join("");

      $("results").innerHTML = `
        <div class="summary-grid">
          <div class="summary-card">
            <div class="k">Kết quả PalmVibe</div>
            <div class="big">${data.summary.overall}/10</div>
            <div class="mid">Vùng nổi bật: ${data.summary.dominant_label}</div>
          </div>

          <div class="summary-card">
            <div class="k">Hướng tay</div>
            <div class="mid">${data.orientation.mode_vi}</div>
            <div style="color:var(--muted);margin-top:10px">${data.orientation.label_vi}</div>
          </div>

          <div class="summary-card">
            <div class="k">Góc xoay</div>
            <div class="mid">${data.rotation_angle}°</div>
            <div style="color:var(--muted);margin-top:10px">Bàn tay đã được căn lại trước khi đọc các vùng chính.</div>
          </div>
        </div>

        <div class="img-grid">
          <div class="img-card">
            <h3>Ảnh gốc</h3>
            <img src="${data.images.original}"/>
          </div>
          <div class="img-card">
            <h3>Vùng lòng bàn tay + 4 vùng crop</h3>
            <img src="${data.images.rotated_overlay}"/>
          </div>
        </div>

        <div class="roi-grid">${cards}</div>

        <div class="viz-grid">
          ${heatmapHTML(data.heatmap)}
          ${radarHTML(data.radar)}
        </div>

        ${renderInterpretation(data)}
        ${renderRelatedPosts(data)}
      `;

      $("results").style.display = "block";
      activateInterpretation(data);
    }
  </script>
</body>
</html>
"""


@app.get("/", response_class=HTMLResponse)
def home():
    html = HTML.replace("__LOGO__", LOGO_HTML)
    html = html.replace("__STATUS__", json.dumps(engine.status()))
    return HTMLResponse(html)

Writing app.py


In [ ]:
!pip install -q --no-cache-dir \
  fastapi \
  uvicorn \
  python-multipart \
  tensorflow==2.20.0 \
  mediapipe \
  opencv-python-headless \
  numpy==1.26.4 \
  pillow==11.3.0 \
  pyngrok

!pkill -f uvicorn || true

from pyngrok import ngrok
import time

NGROK_TOKEN = "3DqD71SQqdUgkYlXSsM2B4InGjy_3BkWt1EBmEyt2CYNfypnn"

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)

get_ipython().system_raw(
    "uvicorn app:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &"
)

time.sleep(10)

public_url = ngrok.connect(8000, "http")
print("MỞ LINK HTTPS NÀY:")
print(public_url)

print("\n===== UVICORN LOG =====")
!tail -80 uvicorn.log

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 166.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 157.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 185.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 223.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 71.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1